# Distributing training across devices

Data parallelism in one line, model parallelism in a DeviceMesh and a LayoutMap — and the sharding you should verify before paying for a long run.

**Runs on:** Multiple GPUs or a TPU · runs single-device with reduced output &nbsp;·&nbsp; **Slides:** [Chapter 18 — Best Practices for the Real World](../../../course-web-slides/ch18/index.html) &nbsp;·&nbsp; **Section:** 03 — Scaling up with multiple devices

---

## What is available

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "jax"     # BEFORE importing keras

import keras
print("backend:", keras.backend.backend())
print("devices:", keras.distribution.list_devices())

The chapter is blunt about the backend: **JAX is the most performant and most scalable, by a mile**, and using anything else for large-scale distributed training wastes compute you are paying for.

## Two kinds of parallelism, for two different problems

In [ ]:
print("DATA PARALLELISM")
print("  one model replicated on every device")
print("  each replica processes a different sub-batch")
print("  gradients averaged; all replicas stay identical (synchronous)")
print("  -> for SPEED. Requires the model to fit on one device.")
print()
print("MODEL PARALLELISM")
print("  one model split across devices, all working on the same batch")
print("  -> for SIZE. Used when the model fits nowhere.")
print()
print("They compose: split across 4, replicate that split twice = 8 devices.")

## Data parallelism is one line

In [ ]:
keras.distribution.set_distribution(keras.distribution.DataParallel())

# ...or naming the devices explicitly:
# keras.distribution.set_distribution(
#     keras.distribution.DataParallel(["gpu:0", "gpu:1"]))

print("set. Nothing else in your code changes.")

> ⚠️ **Before creating the model.** Setting the distribution afterwards silently does nothing — the same ordering trap as `set_dtype_policy` and `KERAS_BACKEND`.

## What to actually expect

In [ ]:
for n, speedup in [(2, 2.0), (4, 3.8), (8, 7.3)]:
    print(f"{n} GPUs -> about {speedup}x   "
          f"({speedup/n:.0%} efficiency)")
print()
print("Merging the weight deltas from different devices takes time,")
print("and the loss grows with device count.")
print()
print("These numbers assume a global batch large enough to keep every")
print("GPU at full capacity. Too small and the speedup collapses.")

## A model too large for one device

In [ ]:
from keras import layers

model = keras.Sequential([
    keras.layers.Input(shape=(16000,)),
    keras.layers.Dense(64000, activation="relu"),
    keras.layers.Dense(8000, activation="sigmoid"),
])
print(f"{model.count_params():,} parameters")
print(f"{model.count_params() * 4 / 1e9:.1f} GB in float32, weights alone")
print()
for v in model.variables:
    print(f"  {v.path:34s} {tuple(v.shape)}")

## The DeviceMesh

In [ ]:
device_mesh = keras.distribution.DeviceMesh(
    shape=(2, 4),
    axis_names=["data", "model"],
)
print(device_mesh)
print()
print("Two devices along axis 0 ('data'): two replicas.")
print("Four along axis 1 ('model'): each replica split across four.")
print("Total: eight devices.")

> **Note** — A mesh need not be 2-D, but in practice you will only ever see 1-D and 2-D. Naming the axes is not decoration — the `LayoutMap` refers to them by name.

## The LayoutMap

In [ ]:
layout_map = keras.distribution.LayoutMap(device_mesh)
layout_map["sequential/dense/kernel"] = (None, "model")
layout_map["sequential/dense/bias"] = ("model",)
layout_map["sequential/dense_1/kernel"] = (None, "model")
layout_map["sequential/dense_1/bias"] = ("model",)

print("None    -> replicate along this dimension")
print("'model' -> shard across the devices of the 'model' mesh axis")
print()
print("Rule of thumb for a simple model:")
print("  shard the LAST dimension along 'model'; replicate everything else.")

In [ ]:
model_parallel = keras.distribution.ModelParallel(
    layout_map=layout_map,
    batch_dim_name="data",
)
keras.distribution.set_distribution(model_parallel)

print("Once set, NOTHING else changes -- the model definition and the")
print("training code are identical, whether you use fit() or your own loop.")

## Verify the sharding before paying for a long run

In [ ]:
# Rebuild under the distribution so the layout takes effect.
model = keras.Sequential([
    keras.layers.Input(shape=(16000,)),
    keras.layers.Dense(64000, activation="relu"),
    keras.layers.Dense(8000, activation="sigmoid"),
])

try:
    print(model.layers[0].kernel.value.sharding)
    import jax
    v = model.layers[0].kernel.value
    jax.debug.visualize_sharding(v.shape, v.sharding)
except Exception as e:
    print("(single-device run -- nothing to shard)", e)

**A silently wrong layout still trains — just slowly, and on the wrong devices.** Print the sharding, or visualise it, before committing to a run you will be billed for.

## The input pipeline, which becomes the bottleneck

In [ ]:
print("Always pass a tf.data.Dataset (NumPy arrays get converted anyway).")
print("Always prefetch:")
print("    dataset = dataset.prefetch(tf.data.AUTOTUNE)")
print()
print("On TPU, also cache if the dataset fits in VM memory:")
print("    dataset = dataset.cache()")
print()
print("A starved input pipeline turns eight expensive GPUs into eight")
print("expensive IDLE GPUs -- and that is the most common way a")
print("distributed run underperforms.")

## TPUs, and step fusing

In [ ]:
print("TPU v2 is free in Colab (Runtime -> Change Runtime Type).")
print("~15x an NVIDIA P100; ~3x more cost-effective than GPU on average.")
print()
print("With the JAX backend, the same set_distribution() call is all")
print("you need -- again, BEFORE creating the model.")
print()
print("Small models underutilize a TPU. Keeping the cores busy can need")
print("batches upward of 10,000 samples, which also means raising the")
print("learning rate: fewer updates, each more accurate.")
print()
print("Or use step fusing, which keeps the batch reasonable:")
print("    model.compile(..., steps_per_execution=8)")

---

## What to take away

- Data parallelism is for speed and needs the model to fit; model parallelism is for size.
- Set the distribution **before** creating the model.
- 8 GPUs give about 7.3×, and only with a large enough global batch.
- Verify the sharding, and prefetch — a starved pipeline is the commonest way a distributed run underperforms.